In [56]:
import pandas as pd 
import numpy as np
import tensorflow as tf 
from tensorflow import keras
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

In [57]:
df = pd.read_csv("diabetes.csv")
df.head(10)

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1
5,5,116,74,0,0,25.6,0.201,30,0
6,3,78,50,32,88,31.0,0.248,26,1
7,10,115,0,0,0,35.3,0.134,29,0
8,2,197,70,45,543,30.5,0.158,53,1
9,8,125,96,0,0,0.0,0.232,54,1


In [58]:
df.shape

(768, 9)

In [59]:
df.columns

Index(['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin',
       'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'],
      dtype='str')

In [60]:
model = keras.Sequential([

    keras.Input(shape=(8,)),

    keras.layers.Dense(16, activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.3),

    keras.layers.Dense(8, activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.3),

    keras.layers.Dense(1, activation='sigmoid')
])

In [61]:
model.compile(
    optimizer = 'adam', 
    loss='binary_crossentropy', 
    metrics=['accuracy']
)

In [62]:
x = df.drop("Outcome", axis=1)
y = df["Outcome"]

In [63]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x,y, test_size=0.2, random_state=42)

In [64]:
checkpoint = ModelCheckpoint(
    "best_model.keras",
    monitor = 'val_loss',
    save_best_only=True,
    verbose=1
)

In [65]:
early_stop = EarlyStopping(
    monitor = 'val_loss',
    patience = 5,
    restore_best_weights = True
)


In [66]:
history = model.fit(
    x_train,
    y_train,
    validation_data = (x_test,y_test),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop, checkpoint]
)

Epoch 1/100
 1/20 ━━━━━━━━━━━━━━━━━━━━ 30s 2s/step - accuracy: 0.4375 - loss: 1.1098
Epoch 1: val_loss improved from None to 1.74589, saving model to best_model.keras

Epoch 1: finished saving model to best_model.keras
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.4430 - loss: 1.0003 - val_accuracy: 0.6429 - val_loss: 1.7459
Epoch 2/100
 1/20 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.4688 - loss: 0.9569
Epoch 2: val_loss improved from 1.74589 to 1.06113, saving model to best_model.keras

Epoch 2: finished saving model to best_model.keras
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5065 - loss: 0.9250 - val_accuracy: 0.6623 - val_loss: 1.0611
Epoch 3/100
 1/20 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.5938 - loss: 0.7573
Epoch 3: val_loss improved from 1.06113 to 0.83599, saving model to best_model.keras

Epoch 3: finished saving model to best_model.keras
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5375 - loss: 0.8604 - val_accuracy: 0.6104 - val_lo

In [67]:
loss, accuracy = model.evaluate(x_test, y_test)

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7727 - loss: 0.5072 


In [68]:
print("Accuracy",accuracy)

Accuracy 0.7727272510528564


In [75]:
predictions = model.predict(x_test)
print(predictions[:5])

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step 
[[0.23861337]
 [0.20008959]
 [0.15406671]
 [0.16026363]
 [0.45284784]]


In [70]:
predicted_labels = (predictions > 0.5).astype(int)
print(predicted_labels[:5])

[[0]
 [0]
 [0]
 [0]
 [0]]


In [71]:
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_10 (Dense)                     │ (None, 16)                  │             144 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_5                │ (None, 16)                  │              64 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_5 (Dropout)                  │ (None, 16)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_11 (Dense)                     │ (None, 8)                   │             136 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_6                │ (None, 8)                   │              32 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_6 (Dropout)                  │ (None, 8)                   │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_12 (Dense)                     │ (None, 1)                   │               9 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,061 (4.15 KB)

 Trainable params: 337 (1.32 KB)

 Non-trainable params: 48 (192.00 B)

 Optimizer params: 676 (2.64 KB)

In [72]:
print("Best Validation Accuracy:",
      max(history.history["val_accuracy"]))

print("Best Validation Loss:",
      min(history.history["val_loss"]))

Best Validation Accuracy: 0.7727272510528564
Best Validation Loss: 0.5071550011634827


In [73]:
print("Epochs trained:", len(history.history["loss"]))

Epochs trained: 51


In [74]:
print("Best Training Accuracy:",
      max(history.history["accuracy"]))

print("Best Validation Accuracy:",
      max(history.history["val_accuracy"]))

Best Training Accuracy: 0.757328987121582
Best Validation Accuracy: 0.7727272510528564
